# Data Cleaning


### 1. Load Libraries

In [43]:
import os
import pandas as pd
import numpy as np

# Set display options for clean inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### 2. Load Raw Datasets

In [44]:
raw_dir = os.path.join('..', 'data', 'raw')
accounts_df = pd.read_csv(os.path.join(raw_dir, 'finlora_accounts.csv'))
transactions_df = pd.read_csv(os.path.join(raw_dir, 'finlora_transactions.csv'))

print(f'Accounts loaded: {len(accounts_df):,} records | Columns: {accounts_df.shape[1]}')
print(f'Transactions loaded: {len(transactions_df):,} records | Columns: {transactions_df.shape[1]}')

Accounts loaded: 7,200 records | Columns: 8
Transactions loaded: 126,000 records | Columns: 24


### 3. Audit Schema and Missing Values
Checks column names, data types, and inspects null value counts across both tables.

In [45]:
print('Accounts Table Null Audit')
print(accounts_df.isna().sum()[accounts_df.isna().sum() > 0])
print(f'Total account nulls: {accounts_df.isna().sum().sum()}')

print('\nTransactions Table Null Audit')
null_counts = transactions_df.isna().sum()
print(null_counts[null_counts > 0])
print(f'Device ID missing: {transactions_df["device_id"].isna().sum():,} ({transactions_df["device_id"].isna().mean()*100:.2f}%)')
print(f'Merchant Name missing: {transactions_df["merchant_name"].isna().sum():,} ({transactions_df["merchant_name"].isna().mean()*100:.2f}%)')

Accounts Table Null Audit
Series([], dtype: int64)
Total account nulls: 0

Transactions Table Null Audit
merchant_name    28610
device_id        14334
is_new_device    14334
dtype: int64
Device ID missing: 14,334 (11.38%)
Merchant Name missing: 28,610 (22.71%)


### 4. Data Type Standardization
Converts date and timestamp columns into datetime objects to support time-series feature extraction.

In [46]:
transactions_df['timestamp'] = pd.to_datetime(transactions_df['timestamp'])
accounts_df['account_created_date'] = pd.to_datetime(accounts_df['account_created_date'])

print(f'Transaction date range: {transactions_df["timestamp"].min()} to {transactions_df["timestamp"].max()}')
print(f'Account creation date range: {accounts_df["account_created_date"].min()} to {accounts_df["account_created_date"].max()}')

Transaction date range: 2025-02-01 00:05:31 to 2026-08-30 23:33:21
Account creation date range: 2021-12-25 00:00:00 to 2026-08-29 00:00:00


### 5. Audit Monetary Amounts & Deduplication
Audits monetary amounts, notes negative amounts representing chargeback reversals or refund credits, and validates unique transaction identifiers.

In [47]:
neg_amounts = (transactions_df['amount'] < 0).sum()
duplicate_txns = transactions_df['transaction_id'].duplicated().sum()
orphan_accounts = (~transactions_df['account_id'].isin(accounts_df['account_id'])).sum()

print(f'Transactions with negative amounts (refunds/reversals): {neg_amounts:,}')
print(f'Duplicate transaction IDs: {duplicate_txns}')
print(f'Orphan transactions with missing account profile: {orphan_accounts}')

# Absolute amount and refund flag
transactions_df['is_negative_amount'] = (transactions_df['amount'] < 0).astype(int)
transactions_df['abs_amount'] = transactions_df['amount'].abs()

Transactions with negative amounts (refunds/reversals): 291
Duplicate transaction IDs: 0
Orphan transactions with missing account profile: 0


### 6. Handle Missing Hardware & Device Flags
Imputes missing `is_new_device` entries with `0` for transactions without device identifiers.

In [48]:
transactions_df['is_new_device'] = transactions_df['is_new_device'].fillna(0).astype(int)
transactions_df['device_id'] = transactions_df['device_id'].fillna('UNKNOWN_DEVICE')
transactions_df['merchant_name'] = transactions_df['merchant_name'].fillna('Unspecified Merchant')

print('Device novelty flags successfully imputed and cast to integer.')
print(transactions_df['is_new_device'].value_counts())

Device novelty flags successfully imputed and cast to integer.
is_new_device
0    122105
1      3895
Name: count, dtype: int64


### 7. Merge Account Profile Metadata
Enriches transaction records with account holder details (`personal_spend_baseline_usd`, `account_created_date`).

In [49]:
# Select supplementary columns from accounts table to prevent duplicate columns
acc_cols_to_merge = ['account_id', 'account_holder_name', 'account_created_date', 'personal_spend_baseline_usd']
merged_df = transactions_df.merge(accounts_df[acc_cols_to_merge], on='account_id', how='left')

print(f'Merged dataset shape: {merged_df.shape}')
merged_df[['transaction_id', 'account_id', 'timestamp', 'channel', 'amount', 'personal_spend_baseline_usd', 'is_fraud']].head(5)

Merged dataset shape: (126000, 29)


,transaction_id,account_id,timestamp,channel,amount,personal_spend_baseline_usd,is_fraud
0,FLR250216186265,FLR-ACC-100000,2025-02-16 14:55:35,Card Not Present,84233.00,63.23,0
1,FLR250423143305,FLR-ACC-100000,2025-04-23 13:35:40,Web Dashboard,9143.33,63.23,0
2,FLR250424154897,FLR-ACC-100000,2025-04-24 12:30:34,Mobile App,20639.00,63.23,0
3,FLR250428101772,FLR-ACC-100000,2025-04-28 14:41:59,Card Not Present,3177.43,63.23,0
4,FLR250628102827,FLR-ACC-100000,2025-06-28 16:15:47,Mobile App,97839.52,63.23,0


### 8. Export Cleaned Dataset

In [50]:
proc_dir = os.path.join('..', 'data', 'processed')
os.makedirs(proc_dir, exist_ok=True)
out_path = os.path.join(proc_dir, 'cleaned_transactions.csv')

merged_df.to_csv(out_path, index=False)
print(f'Cleaned transaction dataset saved to: {out_path}')
print(f'Total records: {len(merged_df):,} | Fraud count: {merged_df["is_fraud"].sum():,} ({merged_df["is_fraud"].mean()*100:.2f}%)')

Cleaned transaction dataset saved to: ..\data\processed\cleaned_transactions.csv
Total records: 126,000 | Fraud count: 3,352 (2.66%)
